# Instance lineage: demo on the running example

Concept: `overview/concepts/Transformations.md`, sections "Instance Lineage" and "Instance Visualisation" (outside this repo).
Code: package `ocelescope_module_location_based_ocpm.lineage`; this notebook only imports and calls its functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(lineage)`) and `docs/resources/lineage.md`.

What the notebook does:
1. loads the transformation roles ($cons$, $prod$) and, for labels, the object types and activities,
2. computes the lineage relations $R$ and the functions $parents$, $children$, $ev_{prod}$, $E_{cons}$, $siblings$,
3. computes lineages $R(hu, dir, gen)$,
4. builds the four lineage graphs of c2 and renders them (with Graphviz if the `graphviz` Python package and the `dot` program are installed, otherwise it prints the DOT source).

In [ ]:
import math
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import load_level_mapping
from ocelescope_module_location_based_ocpm.lineage.graph import lineage_graph, to_dot
from ocelescope_module_location_based_ocpm.lineage.lineage import lineage
from ocelescope_module_location_based_ocpm.lineage.relations import (
    children,
    consumption_events,
    creation_events,
    lineage_relations,
    parents,
    siblings,
)

EXAMPLE = Path("..") / "data" / "running_example"
INPUTS = EXAMPLE / "inputs"

## 1. Inputs

### Transformation roles

`inputs/transformation_roles.csv`: one row per consumed or produced handling unit of a transformation event, columns event, hu, role (`consumed` or `produced`); the same as the e2o qualifiers of the log (option 1 of the note).
A handling unit consumed and produced at the same event has two rows.

In [ ]:
roles = pl.read_csv(INPUTS / "transformation_roles.csv")
roles

### Labels

Object types and activities for the graph labels, from the location mapping saved by `location_derivation.ipynb` (any level has them).

In [ ]:
mapping = load_level_mapping(EXAMPLE / "derived", 3)
object_types = dict(mapping.select("hu", "object_type").unique().iter_rows())
activities = dict(mapping.select("event", "activity").unique().iter_rows())
object_types, activities

## 2. Lineage relations and functions

### `lineage_relations(roles)`

$R = \{(hu, hu', e) \mid e \in T: hu \in cons(e), hu' \in prod(e)\}$: a join of the consumed (parent) and produced (child) handling units on the event.
e1 produces s1 from nothing, so it gives no relation.

In [ ]:
relations = lineage_relations(roles)
relations

### `parents(relations)`, `children(relations)`

$parents(hu) = \{hu' \mid \exists e: (hu', hu, e) \in R\}$ and $children(hu) = \{hu' \mid \exists e: (hu, hu', e) \in R\}$, as relation tables (hu, parent) and (hu, child).

In [ ]:
parents(relations)

In [ ]:
children(relations)

### `creation_events(roles)`, `consumption_events(roles)`

$ev_{prod}(hu)$ (the event producing $hu$; a function under AS-T2 and AS-T4) and $E_{cons}(hu) = \{e \in T \mid hu \in cons(e)\}$, as tables (hu, event).
They use the roles, not $R$: s1 has a creation event (e1) but no parents.

In [ ]:
creation_events(roles)

In [ ]:
consumption_events(roles)

### `siblings(roles)`

$siblings(hu) = prod(ev_{prod}(hu)) \setminus \{hu\}$: the handling units produced in the same event, as a table (hu, sibling, event).

In [ ]:
siblings(roles)

## 3. Lineage: `lineage(relations, hu, direction="both", gen=math.inf)`

$R(hu, dir, gen)$: the relations of the components ($dir$ = `components`), the products (`products`) or both, up to `gen` generations per direction.
How it works: generation by generation (`generations`), each one a filter of $R$ on the handling units reached in the previous generation; it stops after `gen` generations or at an empty generation. A relation reached again in a later generation keeps its first generation.

| parameter | meaning | default |
|---|---|---|
| `relations` | the lineage relations $R$ | |
| `hu` | the handling unit | |
| `direction` | `components`, `products` or `both` | `both` |
| `gen` | generations per direction, a natural number or `math.inf` | `math.inf` |

In [ ]:
lineage(relations, "c2", "both", 1)

In [ ]:
lineage(relations, "s1", "products")

In [ ]:
lineage(relations, "c4", "components")

## 4. Graphs: `lineage_graph(...)` and `to_dot(...)`

`lineage_graph(relations, roles, hu, direction="both", gen=math.inf, with_events=False, with_siblings=False)` returns the nodes (node, kind, role) and edges (source, target, kind, event, directed) of one of the four graphs of the note:

| `with_events` | `with_siblings` | graph |
|---|---|---|
| False | False | object graph: handling units, arcs component → product labelled with the event |
| False | True | plus the siblings $S$, joined by undirected sibling edges |
| True | False | bipartite graph: arcs component → event → product |
| True | True | plus the siblings, with arcs from their creation event |

`to_dot(graph, object_types=None, activities=None, rankdir="LR")` writes Graphviz DOT source: hus filled by object type, events as white boxes, siblings greyed and dashed, the selected hu with a thick border.

In [ ]:
graph = lineage_graph(relations, roles, "c2", with_events=True, with_siblings=True)
graph.nodes

In [ ]:
graph.edges

In [ ]:
def show(dot: str):
    """Render DOT source with Graphviz if the package and the `dot` program are installed, else print it."""
    import shutil

    try:
        import graphviz
    except ImportError:
        graphviz = None
    if graphviz is not None and shutil.which("dot"):
        return graphviz.Source(dot)
    print(dot)


for with_events in (False, True):
    for with_siblings in (False, True):
        print(f"with_events={with_events}, with_siblings={with_siblings}")
        g = lineage_graph(relations, roles, "c2", with_events=with_events, with_siblings=with_siblings)
        display(show(to_dot(g, object_types, activities)))